# Phân khúc khách hàng

Notebook trình bày các bước chuẩn bị dữ liệu RFM, so sánh bốn phương pháp phân khúc và đọc hồ sơ từng nhóm khách hàng. Kết quả được lấy từ các bảng đã lưu trong `reports/`.

Để chạy lại thực nghiệm và cập nhật các bảng kết quả, dùng lệnh sau từ thư mục dự án.

```bash
python scripts/run_segmentation.py --expected-rows 234359
```

## 1. Đọc bảng RFM

Kiểm tra số khách và xem phân bố R, F, M trước khi chuẩn hóa.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.segmentation.features import FEATURE_COLUMNS, load_rfm, prepare_features

rfm = load_rfm(ROOT / "data/processed/customer_rfm.parquet", expected_rows=234_359)
display(rfm.head())
display(rfm[list(FEATURE_COLUMNS)].describe().round(2))

## 2. Tạo đặc trưng cho mô hình

Baseline xếp nhóm theo ngưỡng trên RFM gốc. K-Means, GMM và BIRCH dùng cùng ma trận sau khi áp dụng `log1p` và `StandardScaler`.

In [ ]:
features = prepare_features(rfm)
scaled = pd.DataFrame(features.transformed, columns=FEATURE_COLUMNS)
display(scaled.head().round(3))

## 3. So sánh bốn phương pháp

Xem cấu hình đã chọn, Silhouette, thời gian trung vị và tỷ trọng cụm nhỏ nhất. Ô thời gian trống cho biết lần chạy đã lưu chưa đo phương pháp đó.

In [ ]:
reports = ROOT / "reports"
comparison = pd.read_csv(reports / "model_comparison.csv")
selected = comparison.loc[comparison["selected"]].copy()
selected["median_seconds"] = selected["median_fit_seconds"].where(
    selected["fit_seconds_runs"] != "[]"
)
selected["smallest_segment_pct"] *= 100
display(selected[[
    "method", "config_id", "actual_clusters", "silhouette",
    "median_seconds", "smallest_segment_pct",
]].round(4))

## 4. Đọc hồ sơ phân khúc

Xem số khách, RFM điển hình và tỷ lệ mua lại của từng nhóm. Đối chiếu Silhouette với quy mô cụm trước khi chọn phương pháp đưa lên dashboard.

In [ ]:
profiles = pd.read_csv(reports / "segment_profiles.csv")
display(profiles.round(3))